# 15 · Infix, Postfix & Prefix Expressions

The same arithmetic expression can be written three ways. They differ only in **where the operator goes**:

| Notation | Operator position | Example for "(x + y) × (a − b)" |
|---|---|---|
| **Infix** | between operands | `(x+y)*(a-b)` |
| **Prefix** (Polish) | **before** its operands | `*+xy-ab` |
| **Postfix** (Reverse Polish) | **after** its operands | `xy+ab-*` |

Infix is what humans read, but it needs **parentheses and precedence rules**. Prefix and postfix need **neither**: the order of the symbols already says what to compute first, which makes them ideal for computers (compilers, calculators).

**Operands:** letters `A–Z`, `a–z` and digits `0–9` (single characters). **Operators:** `+  -  *  /  ^`

**Precedence (high → low)** and **associativity**

| Operator | Precedence | Associativity |
|---|---|---|
| `^` (power) | 3 | **right**: `a^b^c = a^(b^c)` |
| `*`  `/` | 2 | left: `a/b*c = (a/b)*c` |
| `+`  `-` | 1 | left: `a-b-c = (a-b)-c` |

**Topics covered**
1. Infix → Postfix
2. Infix → Prefix
3. Postfix → Infix, Prefix → Infix
4. Postfix → Prefix, Prefix → Postfix
5. Evaluating postfix / prefix, and a randomised self-test of every converter

## 1. Infix → Postfix (the Shunting-Yard idea)

Scan the infix expression left to right with an **operator stack** and an **output list**:

| Symbol | Action |
|---|---|
| operand | append to output |
| `(` | push on the stack |
| `)` | pop operators to the output **until `(`**, then discard the `(` |
| operator | pop operators from the stack that must run **before** this one, then push it |

"Must run before" means the stack top has **higher precedence**, or **equal precedence and the new operator is left-associative**. For `^` (right-associative) equal precedence does *not* pop.
At the end, pop everything left on the stack.

```
a+b*c   →   a b c * +
```
`*` binds tighter than `+`, so when `*` arrives, `+` stays on the stack and `*` is placed above it; `*` is popped (and printed) first.

> 🐞 **Fix:** the earlier code popped on `>=` for *every* operator, which treats `^` as left-associative and gives `ab^c^` for `a^b^c`. The correct postfix is `abc^^` (compute `b^c` first).

In [1]:
def precedence(op):
    if op in "+-":
        return 1
    if op in "*/":
        return 2
    if op == "^":
        return 3
    return 0                                    # '(' and anything else

def is_operand(ch):
    return ch.isalnum()

def infix_to_postfix(s):
    stack, out = [], []
    for ch in s:
        if is_operand(ch):
            out.append(ch)
        elif ch == "(":
            stack.append(ch)
        elif ch == ")":
            while stack and stack[-1] != "(":
                out.append(stack.pop())
            stack.pop()                          # discard "("
        else:                                    # operator
            while stack and stack[-1] != "(" and (
                    precedence(stack[-1]) > precedence(ch)
                    or (precedence(stack[-1]) == precedence(ch) and ch != "^")):   # "^" is right-assoc
                out.append(stack.pop())
            stack.append(ch)
    while stack:
        out.append(stack.pop())
    return "".join(out)

print(infix_to_postfix("(x+y)*(a-b)"))           # xy+ab-*
print(infix_to_postfix("a+b*c-d"))               # abc*+d-
print(infix_to_postfix("a^b^c"))                 # abc^^   (right-associative)
print(infix_to_postfix("a-b-c"))                 # ab-c-   (left-associative)

xy+ab-*
abc*+d-
abc^^
ab-c-


### Watch the stack work
A tracing version for `a+b*c-d` prints the stack and output after every symbol.

In [2]:
def infix_to_postfix_trace(s):
    stack, out = [], []
    print(f"{'symbol':8}{'stack':12}{'output'}")
    for ch in s:
        if is_operand(ch):
            out.append(ch)
        elif ch == "(":
            stack.append(ch)
        elif ch == ")":
            while stack and stack[-1] != "(":
                out.append(stack.pop())
            stack.pop()
        else:
            while stack and stack[-1] != "(" and (
                    precedence(stack[-1]) > precedence(ch)
                    or (precedence(stack[-1]) == precedence(ch) and ch != "^")):
                out.append(stack.pop())
            stack.append(ch)
        print(f"{ch:8}{''.join(stack):12}{''.join(out)}")
    while stack:
        out.append(stack.pop())
    print(f"{'end':8}{'':12}{''.join(out)}")

infix_to_postfix_trace("a+b*c-d")

symbol  stack       output
a                   a
+       +           a
b       +           ab
*       +*          ab
c       +*          abc
-       -           abc*+
d       -           abc*+d
end                 abc*+d-


## 2. Infix → Prefix

**Trick:** a prefix expression is the mirror image of a postfix expression. So:
1. **Reverse** the infix string.
2. **Swap** every `(` with `)` (because reversing turned them around).
3. Run the infix → postfix algorithm, but with the **opposite tie-break rule**: pop only when the stack top has **strictly higher** precedence. (For `^` equal precedence *does* pop.)
4. **Reverse** the result.

Why the tie-break flips: reading right-to-left turns left-associative operators into right-associative ones and vice versa.

```
a+b+c  → reverse c+b+a → postfix cba++ → reverse ++abc   ✔ ((a+b)+c)
```

In [3]:
def infix_to_prefix(s):
    swap = {"(": ")", ")": "("}
    s = "".join(swap.get(ch, ch) for ch in reversed(s))   # steps 1 + 2
    stack, out = [], []
    for ch in s:                                          # step 3
        if is_operand(ch):
            out.append(ch)
        elif ch == "(":
            stack.append(ch)
        elif ch == ")":
            while stack and stack[-1] != "(":
                out.append(stack.pop())
            stack.pop()
        else:
            while stack and stack[-1] != "(" and (
                    precedence(stack[-1]) > precedence(ch)
                    or (precedence(stack[-1]) == precedence(ch) and ch == "^")):   # flipped tie-break
                out.append(stack.pop())
            stack.append(ch)
    while stack:
        out.append(stack.pop())
    return "".join(reversed(out))                         # step 4

print(infix_to_prefix("(x+y)*(a-b)"))            # *+xy-ab
print(infix_to_prefix("a+b+c"))                  # ++abc
print(infix_to_prefix("a-b-c"))                  # --abc
print(infix_to_prefix("a^b^c"))                  # ^a^bc

*+xy-ab
++abc
--abc
^a^bc


## 3. Postfix → Infix and Prefix → Infix

Use a **stack of sub-expressions**.

**Postfix → Infix** (scan left → right)
- operand → push
- operator → pop `operand2` (top), then `operand1`, push `"(" + operand1 + op + operand2 + ")"`

**Prefix → Infix** (scan **right → left**)
- operand → push
- operator → pop `operand1` (top) first, then `operand2`, push `"(" + operand1 + op + operand2 + ")"`

The order of popping is the only difference: in postfix the **last** operand is on top; in prefix scanning backwards, the **first** operand is on top.

The results are **fully parenthesised**, so there is no ambiguity.

In [4]:
def postfix_to_infix(s):
    stack = []
    for ch in s:
        if is_operand(ch):
            stack.append(ch)
        else:
            right = stack.pop()                  # popped first → second operand
            left = stack.pop()
            stack.append(f"({left}{ch}{right})")
    return stack[-1]

def prefix_to_infix(s):
    stack = []
    for ch in reversed(s):                       # right → left
        if is_operand(ch):
            stack.append(ch)
        else:
            left = stack.pop()                   # popped first → first operand
            right = stack.pop()
            stack.append(f"({left}{ch}{right})")
    return stack[-1]

print(postfix_to_infix("xy+ab-*"))               # ((x+y)*(a-b))
print(prefix_to_infix("*+xy-ab"))                # ((x+y)*(a-b))
print(postfix_to_infix("abc^^"))                 # (a^(b^c))

((x+y)*(a-b))
((x+y)*(a-b))
(a^(b^c))


## 4. Postfix ⇄ Prefix directly

No need to go through infix. Keep a stack of **sub-expressions in the target notation**.

| Conversion | Scan | When an operator arrives (pop `a` then `b`) |
|---|---|---|
| Postfix → Prefix | left → right, pop `right` then `left` | push `op + left + right` |
| Prefix → Postfix | right → left, pop `left` then `right` | push `left + right + op` |

In [5]:
def postfix_to_prefix(s):
    stack = []
    for ch in s:
        if is_operand(ch):
            stack.append(ch)
        else:
            right = stack.pop()
            left = stack.pop()
            stack.append(ch + left + right)      # operator goes in front
    return stack[-1]

def prefix_to_postfix(s):
    stack = []
    for ch in reversed(s):
        if is_operand(ch):
            stack.append(ch)
        else:
            left = stack.pop()
            right = stack.pop()
            stack.append(left + right + ch)      # operator goes at the back
    return stack[-1]

print(postfix_to_prefix("xy+ab-*"))              # *+xy-ab
print(prefix_to_postfix("*+xy-ab"))              # xy+ab-*
assert prefix_to_postfix(postfix_to_prefix("abc*+d-")) == "abc*+d-"     # round trip

*+xy-ab
xy+ab-*


## 5. Evaluate & self-test

To prove the converters are correct, evaluate the expressions numerically.

**Evaluate postfix** (left → right): push numbers; on an operator pop two values, compute `left op right`, push the result.
**Evaluate prefix** (right → left): same, but the first value popped is the **left** operand.

We use `Fraction` for exact arithmetic (no floating-point rounding surprises).

In [6]:
from fractions import Fraction

def apply(op, a, b):
    # lambdas, so that ONLY the requested operation is computed (a dict of plain values would
    # compute a / b even for "+", and crash when b == 0)
    return {"+": lambda: a + b, "-": lambda: a - b, "*": lambda: a * b,
            "/": lambda: a / b, "^": lambda: a ** b}[op]()

def evaluate_postfix(s):
    stack = []
    for ch in s:
        if ch.isdigit():
            stack.append(Fraction(int(ch)))
        else:
            b = stack.pop()
            a = stack.pop()
            stack.append(apply(ch, a, b))
    return stack[0]

def evaluate_prefix(s):
    stack = []
    for ch in reversed(s):
        if ch.isdigit():
            stack.append(Fraction(int(ch)))
        else:
            a = stack.pop()                      # first popped = left operand
            b = stack.pop()
            stack.append(apply(ch, a, b))
    return stack[0]

print(evaluate_postfix("23*54*+9-"))             # 2*3 + 5*4 - 9 = 17
print(evaluate_prefix("-+*23*549"))              # same expression in prefix → 17
print(evaluate_postfix("232^^"))                 # 2^(3^2) = 512

17
17
512


### Randomised self-test
Generate hundreds of random expressions with **minimal parentheses** (so precedence and associativity really matter), then check that
1. the infix → postfix → value, and
2. the infix → prefix → value

both equal Python's own evaluation of the same infix string, and that every other converter round-trips.

In [7]:
import random, re

def random_tree(depth, caret_budget):
    """Random expression tree of single digits. '^' always gets a small single-digit exponent."""
    if depth == 0 or random.random() < 0.25:
        return str(random.randint(1, 9))
    op = random.choice("+-*/^") if caret_budget[0] > 0 else random.choice("+-*/")
    if op == "^":
        caret_budget[0] -= 1
        return (op, random_tree(depth - 1, caret_budget), str(random.randint(1, 3)))
    return (op, random_tree(depth - 1, caret_budget), random_tree(depth - 1, caret_budget))

def to_infix(node, parent_op=None, side=None):
    if isinstance(node, str):
        return node
    op, left, right = node
    text = to_infix(left, op, "L") + op + to_infix(right, op, "R")
    need = False
    if parent_op is not None:
        pp, cp = precedence(parent_op), precedence(op)
        if cp < pp:
            need = True
        elif cp == pp:
            # equal precedence: left-assoc ops need parens on the RIGHT child, '^' needs them on the LEFT child
            need = (side == "R") if parent_op != "^" else (side == "L")
        if random.random() < 0.15:               # sometimes add harmless extra parentheses
            need = True
    return f"({text})" if need else text

def python_value(infix):
    expr = re.sub(r"(\d)", r"F(\1)", infix).replace("^", "**")
    return eval(expr, {"F": Fraction})

random.seed(7)
checked = 0
while checked < 600:
    tree = random_tree(depth=4, caret_budget=[2])
    infix = to_infix(tree)
    try:
        expected = python_value(infix)
    except ZeroDivisionError:
        continue
    post, pre = infix_to_postfix(infix), infix_to_prefix(infix)
    assert evaluate_postfix(post) == expected, (infix, post)
    assert evaluate_prefix(pre) == expected, (infix, pre)
    assert postfix_to_prefix(post) == pre and prefix_to_postfix(pre) == post, infix
    assert python_value(postfix_to_infix(post)) == expected, infix
    assert python_value(prefix_to_infix(pre)) == expected, infix
    checked += 1
print(f"✅ {checked} random expressions: all six converters agree with Python")

✅ 600 random expressions: all six converters agree with Python


---
### ✅ Key takeaways
- **Postfix/prefix need no parentheses**, which is why compilers convert to them.
- Infix → postfix: operand → output, `(` push, `)` pop to `(`, operator → pop higher-or-equal (left-assoc) precedence first.
- Infix → prefix = reverse, swap brackets, postfix algorithm with the **tie-break flipped**, reverse again.
- Postfix and prefix conversions differ only in **scan direction** and **pop order**.
- Always test converters by **evaluating** the result: it catches precedence and associativity bugs.